# p1-img1024-f0

Matched 10-epoch fold-zero screen. It cannot promote directly to the final model and creates no submission.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('ultralytics') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', 'ultralytics==8.4.147', 'ultralytics-thop'], check=True)


In [ ]:
from pathlib import Path
Path('/kaggle/working/hsi_runner.py').write_text('"""Reproducible 16-band hyperspectral detector experiments and submission builder.\n\nSimple explanation:\n\nThis script:\n1. Reads the 16-band hyperspectral images.\n2. Reads the correct object boxes from XML files.\n3. Keeps some training images hidden for validation and holdout testing.\n4. Converts the data into the format YOLO expects.\n5. Trains one YOLO26m or RT-DETR-L object-detection model.\n6. Uses that model on the competition test images.\n7. Creates submission.csv only when holdout mAP50-95 is at least 0.50.\n\nImportant terms:\n- YOLO26m: an object detector. It predicts class, confidence, and box.\n- 16 bands: the 16 spectral channels in each competition image.\n- Validation: hidden training images used to check model progress.\n- Holdout: another hidden local set used as a safer final check.\n- Confidence: how sure the model is about one prediction.\n- Bounding box: rectangle around an object.\n\nThis version can read the competition raw PNG mosaics directly and applies the organizer-provided X2Cube conversion internally. It supports controlled aspect-ratio, cross-validation, spectral-augmentation, and same-checkpoint TTA experiments.\n"""\n\nfrom pathlib import Path\nimport argparse\nimport json\nimport random\nimport hashlib\nimport warnings\nimport importlib.metadata\nimport xml.etree.ElementTree as ET\n\nimport cv2\nfrom PIL import Image\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom sklearn.model_selection import StratifiedGroupKFold, train_test_split\n\n\n\ndef X2Cube(img, cellSize=4):\n    if img.ndim != 2 or any(n % cellSize for n in img.shape):\n        raise ValueError("Raw image sides must be divisible by 4")\n    B = [cellSize, cellSize]\n    skip = [cellSize, cellSize]\n    M, N = img.shape\n    col_extent = N - B[1] + 1\n    row_extent = M - B[0] + 1\n    start_idx = np.arange(B[0])[:, None] * N + np.arange(B[1])\n    didx = M * N * np.arange(1)\n    start_idx = (didx[:, None] + start_idx.ravel()).reshape((-1, B[0], B[1]))\n    offset_idx = np.arange(row_extent)[:, None] * N + np.arange(col_extent)\n    out = np.take(\n        img,\n        start_idx.ravel()[:, None] + offset_idx[::skip[0], ::skip[1]].ravel(),\n    )\n    out = np.transpose(out)\n    return out.reshape(M // cellSize, N // cellSize, cellSize * cellSize)\n\n\ndef read_classes(path):\n    names = [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]\n    if len(names) != 18:\n        raise ValueError(f"Expected 18 classes, found {len(names)}")\n    return names\n\n\ndef read_cube(path):\n    path = Path(path)\n    if path.suffix.lower() == ".png":\n        raw = np.array(Image.open(path))\n        if raw.ndim != 2:\n            raise ValueError(f"{path} must be a 2D raw mosaic PNG, got {raw.shape}")\n        x = X2Cube(raw)\n    elif path.suffix.lower() == ".npy":\n        x = np.load(path)\n    elif path.suffix.lower() == ".npz":\n        z = np.load(path)\n        x = z[z.files[0]]\n    elif path.suffix.lower() in {".tif", ".tiff"}:\n        ok, frames = cv2.imreadmulti(str(path), flags=cv2.IMREAD_UNCHANGED)\n        if not ok:\n            raise ValueError(f"Cannot read {path}")\n        x = np.stack(frames, axis=-1)\n    else:\n        raise ValueError(f"Unsupported cube format: {path.suffix}")\n    x = np.asarray(x)\n    if not np.isfinite(x).all():\n        raise ValueError(f"Non-finite image values in {path}")\n    if x.ndim != 3:\n        raise ValueError(f"{path} must be 3D, got {x.shape}")\n    if x.shape[-1] == 16:\n        return x.astype(np.float32)\n    if x.shape[0] == 16:\n        return np.moveaxis(x, 0, -1).astype(np.float32)\n    raise ValueError(f"{path} does not contain 16 bands: {x.shape}")\n\n\ndef collect_cubes(root):\n    root = Path(root)\n    files = []\n    for ext in ("*.png", "*.npy", "*.npz", "*.tif", "*.tiff"):\n        files.extend(root.rglob(ext))\n    files = sorted({p.resolve() for p in files})\n    if not files:\n        raise FileNotFoundError(\n            "No 16-band cubes found. Run the organizer-compatible X2Cube conversion first."\n        )\n    return files\n\n\ndef band_limits(files, samples_per_image=256, seed=42):\n    rng = np.random.default_rng(seed)\n    chunks = []\n    for p in files:\n        x = read_cube(p).reshape(-1, 16)\n        n = min(samples_per_image, len(x))\n        idx = rng.choice(len(x), n, replace=False)\n        chunks.append(x[idx])\n    sample = np.concatenate(chunks, axis=0)\n    lo = np.percentile(sample, 0.5, axis=0)\n    hi = np.percentile(sample, 99.5, axis=0)\n    hi = np.maximum(hi, lo + 1e-6)\n    return lo, hi\n\n\ndef model_geometry(height, width, input_policy):\n    if input_policy == "native":\n        return {\n            "original_height": height,\n            "original_width": width,\n            "model_height": height,\n            "model_width": width,\n            "pad_top": 0,\n            "pad_left": 0,\n        }\n    if input_policy != "square_pad":\n        raise ValueError(f"Unknown input policy: {input_policy}")\n    side = max(height, width)\n    return {\n        "original_height": height,\n        "original_width": width,\n        "model_height": side,\n        "model_width": side,\n        "pad_top": (side - height) // 2,\n        "pad_left": (side - width) // 2,\n    }\n\n\ndef resolve_input_policy(family, input_policy):\n    if input_policy != "auto":\n        return input_policy\n    return "square_pad" if family == "rtdetr" else "native"\n\n\ndef prepare_cube(cube, lo, hi, input_policy, pipeline_state=None):\n    if pipeline_state is None:\n        x = np.clip((cube - lo) / (hi - lo), 0, 1)\n        x = np.rint(x * 255).astype(np.uint8)\n    else:\n        try:\n            from Experiments.preprocessing.pipeline import apply_pipeline\n        except ModuleNotFoundError:\n            from pipeline import apply_pipeline\n        x = apply_pipeline(cube, pipeline_state, output="uint8")\n    height, width = x.shape[:2]\n    geometry = model_geometry(height, width, input_policy)\n    if input_policy == "square_pad":\n        padded = np.zeros(\n            (geometry["model_height"], geometry["model_width"], 16),\n            dtype=np.uint8,\n        )\n        top = geometry["pad_top"]\n        left = geometry["pad_left"]\n        padded[top : top + height, left : left + width] = x\n        x = padded\n    return x, geometry\n\n\ndef write_tiff(cube, path, lo, hi, input_policy, pipeline_state=None):\n    x, geometry = prepare_cube(cube, lo, hi, input_policy, pipeline_state)\n    chw = np.moveaxis(x, -1, 0)\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    ok = cv2.imwritemulti(str(path), chw)\n    if not ok:\n        raise IOError(f"Failed to write {path}")\n    return geometry\n\n\ndef transform_rows(rows, geometry):\n    source_width = geometry["original_width"]\n    source_height = geometry["original_height"]\n    target_width = geometry["model_width"]\n    target_height = geometry["model_height"]\n    pad_left = geometry["pad_left"]\n    pad_top = geometry["pad_top"]\n    transformed = []\n    for cls, cx, cy, bw, bh in rows:\n        transformed.append(\n            (\n                cls,\n                (cx * source_width + pad_left) / target_width,\n                (cy * source_height + pad_top) / target_height,\n                bw * source_width / target_width,\n                bh * source_height / target_height,\n            )\n        )\n    return transformed\n\n\ndef restore_boxes(boxes, geometry):\n    boxes = np.asarray(boxes, dtype=np.float32).copy()\n    boxes[:, [0, 2]] -= geometry["pad_left"]\n    boxes[:, [1, 3]] -= geometry["pad_top"]\n    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, geometry["original_width"])\n    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, geometry["original_height"])\n    return boxes\n\n\ndef spectral_augment(cube, rng, gain_limit, tilt_limit, noise_std):\n    cube = cube.astype(np.float32, copy=True)\n    gain = rng.uniform(1.0 - gain_limit, 1.0 + gain_limit)\n    tilt = rng.uniform(-tilt_limit, tilt_limit)\n    wavelength_axis = np.linspace(-1.0, 1.0, 16, dtype=np.float32)\n    cube *= gain * (1.0 + tilt * wavelength_axis[None, None, :])\n    if noise_std > 0:\n        scale = np.maximum(cube.std(axis=(0, 1)), 1e-6)\n        noise = rng.normal(0.0, noise_std, size=cube.shape).astype(np.float32)\n        cube += noise * scale[None, None, :]\n    return cube\n\n\ndef stable_rng(seed, stem, copy_index):\n    digest = hashlib.sha256(f"{seed}:{stem}:{copy_index}".encode()).digest()\n    return np.random.default_rng(int.from_bytes(digest[:8], "little"))\n\n\ndef expand_rgb_stem(target_model, source_model):\n    import torch\n\n    target_layers = [layer for layer in target_model.modules() if isinstance(layer, torch.nn.Conv2d)]\n    source_layers = [layer for layer in source_model.modules() if isinstance(layer, torch.nn.Conv2d)]\n    target = next((layer for layer in target_layers if layer.in_channels == 16), None)\n    source = next(\n        (\n            layer\n            for layer in source_layers\n            if layer.in_channels == 3\n            and target is not None\n            and layer.out_channels == target.out_channels\n            and layer.kernel_size == target.kernel_size\n        ),\n        None,\n    )\n    if target is None or source is None:\n        raise RuntimeError("Could not identify compatible 3-channel and 16-channel input stems")\n    expanded = source.weight.detach().mean(dim=1, keepdim=True).repeat(1, 16, 1, 1)\n    expanded *= 3.0 / 16.0\n    target.weight.data.copy_(expanded.to(device=target.weight.device, dtype=target.weight.dtype))\n    if target.bias is not None and source.bias is not None:\n        target.bias.data.copy_(source.bias.detach().to(device=target.bias.device, dtype=target.bias.dtype))\n    print("Initialized the 16-channel input stem from RGB weights with activation-scale correction")\n\n\ndef trainer_class(args):\n    if args.stem_init == "random":\n        return None\n    if args.family == "rtdetr":\n        from ultralytics.models.rtdetr.train import RTDETRTrainer\n\n        class HSITrainer(RTDETRTrainer):\n            def get_model(self, cfg=None, weights=None, verbose=True):\n                model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n                if weights is None:\n                    raise RuntimeError("rgb_mean stem initialization requires pretrained weights")\n                expand_rgb_stem(model, weights)\n                return model\n\n        return HSITrainer\n    from ultralytics.models.yolo.detect import DetectionTrainer\n\n    class HSITrainer(DetectionTrainer):\n        def get_model(self, cfg=None, weights=None, verbose=True):\n            model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n            if weights is None:\n                raise RuntimeError("rgb_mean stem initialization requires pretrained weights")\n            expand_rgb_stem(model, weights)\n            return model\n\n    return HSITrainer\n\n\ndef parse_voc(xml_path, class_to_id):\n    root = ET.parse(xml_path).getroot()\n    size = root.find("size")\n    width = int(float(size.findtext("width")))\n    height = int(float(size.findtext("height")))\n    rows = []\n    for obj in root.findall("object"):\n        name = obj.findtext("name").strip()\n        if name not in class_to_id:\n            raise ValueError(f"Unknown class {name} in {xml_path}")\n        b = obj.find("bndbox")\n        x1 = float(b.findtext("xmin"))\n        y1 = float(b.findtext("ymin"))\n        x2 = float(b.findtext("xmax"))\n        y2 = float(b.findtext("ymax"))\n        x1 = min(max(x1, 0.0), width)\n        x2 = min(max(x2, 0.0), width)\n        y1 = min(max(y1, 0.0), height)\n        y2 = min(max(y2, 0.0), height)\n        if x2 <= x1 or y2 <= y1:\n            continue\n        cx = ((x1 + x2) / 2) / width\n        cy = ((y1 + y2) / 2) / height\n        bw = (x2 - x1) / width\n        bh = (y2 - y1) / height\n        rows.append((class_to_id[name], cx, cy, bw, bh))\n    return width, height, rows\n\n\ndef annotation_map(xml_root):\n    xmls = list(Path(xml_root).rglob("*.xml"))\n    if not xmls:\n        raise FileNotFoundError(f"No VOC XML files found under {xml_root}")\n    return unique_stems(xmls)\n\n\ndef rarest_image_label(stems, boxes_by_stem):\n    counts = {}\n    for stem in stems:\n        for row in boxes_by_stem[stem]:\n            cls = int(row[0])\n            counts[cls] = counts.get(cls, 0) + 1\n    labels = []\n    for stem in stems:\n        classes = {int(row[0]) for row in boxes_by_stem[stem]}\n        if not classes:\n            labels.append(-1)\n        else:\n            labels.append(min(classes, key=lambda c: counts[c]))\n    return labels\n\n\ndef safe_split(stems, boxes_by_stem, seed):\n    labels = rarest_image_label(stems, boxes_by_stem)\n    try:\n        train, temp = train_test_split(\n            stems,\n            test_size=0.20,\n            random_state=seed,\n            stratify=labels,\n        )\n        temp_labels = rarest_image_label(temp, boxes_by_stem)\n        val, holdout = train_test_split(\n            temp,\n            test_size=0.50,\n            random_state=seed,\n            stratify=temp_labels,\n        )\n    except ValueError as exc:\n        warnings.warn(f"Class-balanced split failed; using random split: {exc}")\n        train, temp = train_test_split(\n            stems,\n            test_size=0.20,\n            random_state=seed,\n            shuffle=True,\n        )\n        val, holdout = train_test_split(\n            temp,\n            test_size=0.50,\n            random_state=seed,\n            shuffle=True,\n        )\n    return sorted(train), sorted(val), sorted(holdout)\n\n\ndef read_groups(path, stems):\n    if not path:\n        return {stem: stem for stem in stems}, "image_id_only"\n    frame = pd.read_csv(path, dtype=str)\n    expected = {"image_id", "group_id"}\n    if set(frame.columns) != expected:\n        raise ValueError(f"Groups file columns must be {sorted(expected)}")\n    if frame["image_id"].duplicated().any():\n        raise ValueError("Groups file contains duplicate image_id values")\n    groups = dict(zip(frame["image_id"], frame["group_id"]))\n    missing = sorted(set(stems) - set(groups))\n    if missing:\n        raise ValueError(f"Groups file is missing {len(missing)} training IDs")\n    return {stem: groups[stem] for stem in stems}, str(Path(path))\n\n\ndef cv_split(stems, boxes_by_stem, groups_by_stem, seed, folds, fold_index):\n    if folds < 2:\n        raise ValueError("cv_folds must be at least 2")\n    if not 0 <= fold_index < folds:\n        raise ValueError(f"cv_fold_index must be in 0..{folds - 1}")\n    labels = np.asarray(rarest_image_label(stems, boxes_by_stem))\n    groups = np.asarray([groups_by_stem[stem] for stem in stems])\n    holdout_splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)\n    design_idx, holdout_idx = next(holdout_splitter.split(stems, labels, groups))\n    design_stems = np.asarray(stems)[design_idx]\n    design_labels = labels[design_idx]\n    design_groups = groups[design_idx]\n    fold_splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed + 1)\n    selected = list(fold_splitter.split(design_stems, design_labels, design_groups))[fold_index]\n    train_idx, val_idx = selected\n    return (\n        sorted(design_stems[train_idx].tolist()),\n        sorted(design_stems[val_idx].tolist()),\n        sorted(np.asarray(stems)[holdout_idx].tolist()),\n    )\n\n\ndef load_split_manifest(path, stems):\n    split_map = json.loads(Path(path).read_text())\n    if set(split_map) != {"train", "val", "test"}:\n        raise ValueError("Split manifest must contain train, val, and test")\n    flattened = [str(stem) for split in split_map.values() for stem in split]\n    if len(flattened) != len(set(flattened)):\n        raise ValueError("Split manifest contains duplicate IDs across splits")\n    if set(flattened) != set(stems):\n        raise ValueError("Split manifest IDs do not exactly match usable training IDs")\n    return {key: sorted(map(str, value)) for key, value in split_map.items()}\n\n\ndef assert_group_isolation(split_map, groups_by_stem):\n    group_splits = {}\n    for split, stems in split_map.items():\n        for stem in stems:\n            group = groups_by_stem[stem]\n            if group in group_splits and group_splits[group] != split:\n                raise ValueError(f"Group {group} leaks across {group_splits[group]} and {split}")\n            group_splits[group] = split\n\n\ndef write_label(path, rows):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    text = "\\n".join(\n        f"{int(c)} {cx:.8f} {cy:.8f} {w:.8f} {h:.8f}"\n        for c, cx, cy, w, h in rows\n    )\n    Path(path).write_text(text + ("\\n" if text else ""))\n\n\ndef image_id_value(stem):\n    return int(stem) if stem.isdigit() else stem\n\n\ndef parse_class_ids(value):\n    if not value:\n        return []\n    ids = sorted({int(item) for item in value.split(",")})\n    if any(class_id < 0 or class_id > 17 for class_id in ids):\n        raise argparse.ArgumentTypeError("Class IDs must be between 0 and 17")\n    return ids\n\n\ndef build_dataset(args):\n    work = Path(args.workdir)\n    work.mkdir(parents=True, exist_ok=True)\n    cache_root = Path(args.cache_dir) if args.cache_dir else work\n    prepared = cache_root / "dataset"\n    if prepared.exists():\n        raise FileExistsError(f"Use a fresh workdir; keeping existing data: {prepared}")\n    names = read_classes(args.class_file)\n    class_to_id = {name: i for i, name in enumerate(names)}\n    train_cubes = collect_cubes(args.train_cubes)\n    design_only = getattr(args, "design_only", False)\n    test_cubes = [] if design_only else collect_cubes(args.test_cubes)\n    ranking_cubes = collect_cubes(args.ranking_cubes) if args.ranking_cubes and not design_only else []\n    xmls = annotation_map(args.train_xml)\n    cube_by_stem = unique_stems(train_cubes)\n    inference_by_stem = unique_stems(test_cubes + ranking_cubes)\n    if set(cube_by_stem) != set(xmls):\n        raise ValueError("Training image and XML IDs do not match")\n    stems = sorted(set(cube_by_stem) & set(xmls))\n    bad_stems = {"1227", "1836", "1855"}\n    stems = [s for s in stems if s not in bad_stems]\n    print(f"Skipping mismatched images: {sorted(bad_stems)}")\n\n    if not stems:\n        raise RuntimeError("No matching training cube/XML stems found")\n\n    boxes_by_stem = {}\n    dims_by_stem = {}\n    for stem in stems:\n        w, h, rows = parse_voc(xmls[stem], class_to_id)\n        boxes_by_stem[stem] = rows\n        dims_by_stem[stem] = (w, h)\n\n    groups_by_stem, groups_source = read_groups(args.groups_file, stems)\n    if args.split_manifest:\n        split_map = load_split_manifest(args.split_manifest, stems)\n    elif args.cv_fold_index is not None:\n        train_stems, val_stems, holdout_stems = cv_split(\n            stems,\n            boxes_by_stem,\n            groups_by_stem,\n            args.seed,\n            args.cv_folds,\n            args.cv_fold_index,\n        )\n        split_map = {"train": train_stems, "val": val_stems, "test": holdout_stems}\n    else:\n        train_stems, val_stems, holdout_stems = safe_split(stems, boxes_by_stem, args.seed)\n        split_map = {"train": train_stems, "val": val_stems, "test": holdout_stems}\n    assert_group_isolation(split_map, groups_by_stem)\n    train_stems = split_map["train"]\n    val_stems = split_map["val"]\n    holdout_stems = split_map["test"]\n    normalization = getattr(args, "normalization", "N0")\n    spectral_preprocess = getattr(args, "spectral_preprocess", "SP0")\n    pipeline_state = None\n    if normalization == "N0" and spectral_preprocess == "SP0":\n        lo, hi = band_limits([cube_by_stem[s] for s in train_stems], seed=args.seed)\n    else:\n        try:\n            from Experiments.preprocessing.pipeline import fit_pipeline, save_pipeline_state\n        except ModuleNotFoundError:\n            from pipeline import fit_pipeline, save_pipeline_state\n        pipeline_state = fit_pipeline(\n            (read_cube(cube_by_stem[s]) for s in train_stems),\n            normalization,\n            spectral_method=spectral_preprocess,\n            seed=args.seed,\n            samples_per_cube=getattr(args, "normalization_samples_per_image", 256),\n            mnf_components=getattr(args, "mnf_components", None),\n            savgol_window=getattr(args, "savgol_window", 5),\n            savgol_order=getattr(args, "savgol_order", 2),\n        )\n        lo = hi = None\n        save_pipeline_state(pipeline_state, work / "spectral_pipeline_state.json")\n    input_policy = resolve_input_policy(args.family, args.input_policy)\n    (work / "split_manifest.json").write_text(json.dumps(split_map, indent=2))\n    train_images_written = 0\n    for split, split_stems in split_map.items():\n        if design_only and split == "test":\n            continue\n        for stem in split_stems:\n            cube = read_cube(cube_by_stem[stem])\n            h, w = cube.shape[:2]\n            ann_w, ann_h = dims_by_stem[stem]\n            if (w, h) != (ann_w, ann_h):\n                raise ValueError(\n                    f"Spatial mismatch for {stem}: cube {(w, h)} vs annotation {(ann_w, ann_h)}"\n                )\n            geometry = write_tiff(\n                cube,\n                prepared / "images" / split / f"{stem}.tiff",\n                lo,\n                hi,\n                input_policy,\n                pipeline_state,\n            )\n            rows = transform_rows(boxes_by_stem[stem], geometry)\n            write_label(prepared / "labels" / split / f"{stem}.txt", rows)\n            if split != "train":\n                continue\n            train_images_written += 1\n            classes = {int(row[0]) for row in boxes_by_stem[stem]}\n            selected = not args.spectral_aug_classes or classes.intersection(args.spectral_aug_classes)\n            if not selected:\n                continue\n            for copy_index in range(args.spectral_aug_copies):\n                suffix = f"{stem}_hsaug{copy_index + 1}"\n                augmented = spectral_augment(\n                    cube,\n                    stable_rng(args.seed, stem, copy_index),\n                    args.spectral_gain,\n                    args.spectral_tilt,\n                    args.spectral_noise,\n                )\n                write_tiff(\n                    augmented,\n                    prepared / "images" / split / f"{suffix}.tiff",\n                    lo,\n                    hi,\n                    input_policy,\n                    pipeline_state,\n                )\n                write_label(prepared / "labels" / split / f"{suffix}.txt", rows)\n                train_images_written += 1\n\n    test_out = cache_root / "test_tiff"\n    if test_out.exists():\n        raise FileExistsError(f"Use a fresh workdir; keeping existing data: {test_out}")\n    test_out.mkdir(parents=True, exist_ok=True)\n    inference_geometry = {}\n    for stem, path in inference_by_stem.items():\n        inference_geometry[stem] = write_tiff(\n            read_cube(path),\n            test_out / f"{stem}.tiff",\n            lo,\n            hi,\n            input_policy,\n            pipeline_state,\n        )\n    (work / "inference_geometry.json").write_text(json.dumps(inference_geometry, indent=2))\n\n    data = {\n        "path": str(prepared),\n        "train": "images/train",\n        "val": "images/val",\n        "test": "images/test",\n        "channels": 16,\n        "names": {i: n for i, n in enumerate(names)},\n    }\n    yaml_path = work / "dataset.yaml"\n    yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))\n\n    report = {\n        "train_images_total": len(stems),\n        "train_split": len(train_stems),\n        "val_split": len(val_stems),\n        "holdout_split": len(holdout_stems),\n        "prepared_train_images": train_images_written,\n        "competition_test_images": len(test_cubes),\n        "competition_ranking_images": len(ranking_cubes),\n        "class_names": names,\n        "excluded_ids": sorted(bad_stems),\n        "split_ids": split_map,\n        "split_mode": "manifest" if args.split_manifest else ("cv" if args.cv_fold_index is not None else "legacy"),\n        "cv_folds": args.cv_folds if args.cv_fold_index is not None else None,\n        "cv_fold_index": args.cv_fold_index,\n        "groups_source": groups_source,\n        "group_count": len(set(groups_by_stem.values())),\n        "input_policy": input_policy,\n        "spectral_augmentation": {\n            "copies": args.spectral_aug_copies,\n            "classes": args.spectral_aug_classes,\n            "gain_limit": args.spectral_gain,\n            "tilt_limit": args.spectral_tilt,\n            "noise_std": args.spectral_noise,\n        },\n        "class_counts": {split: np.bincount([int(row[0]) for stem in ids for row in boxes_by_stem[stem]], minlength=18).tolist() for split, ids in split_map.items()},\n        "band_low_0_5pct": lo.tolist() if lo is not None else None,\n        "band_high_99_5pct": hi.tolist() if hi is not None else None,\n        "normalization": normalization,\n        "spectral_preprocess": spectral_preprocess,\n        "spectral_pipeline_state": pipeline_state.to_dict() if pipeline_state is not None else None,\n    }\n    (work / "data_audit.json").write_text(json.dumps(report, indent=2))\n    return yaml_path, test_out, inference_geometry\n\n\ndef train(args, yaml_path):\n    model = model_class(args)(args.model)\n    result = model.train(\n        trainer=trainer_class(args),\n        data=str(yaml_path),\n        epochs=args.epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        workers=args.workers,\n        project=str(Path(args.workdir) / "runs"),\n        name=args.run_name,\n        seed=args.seed,\n        deterministic=args.family == "yolo",\n        amp=args.family == "yolo",\n        rect=args.family == "yolo",\n        multi_scale=args.multi_scale,\n        patience=25,\n        close_mosaic=15,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        degrees=5.0,\n        translate=0.08,\n        scale=0.35,\n        fliplr=0.5,\n        flipud=0.0,\n        mosaic=0.5,\n        mixup=0.0,\n        copy_paste=0.0,\n        augmentations=[],\n        plots=False,\n        save=True,\n        verbose=True,\n        pretrained=True,\n    )\n    best = Path(result.save_dir) / "weights" / "best.pt"\n    if not best.exists():\n        raise FileNotFoundError(best)\n    return best\n\n\ndef localization_finetune(args, yaml_path, checkpoint):\n    if args.localization_epochs <= 0:\n        return None\n    model = model_class(args)(str(checkpoint))\n    result = model.train(\n        data=str(yaml_path),\n        epochs=args.localization_epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        workers=args.workers,\n        project=str(Path(args.workdir) / "runs"),\n        name=f"{args.run_name}_localize",\n        seed=args.seed,\n        deterministic=args.family == "yolo",\n        amp=args.family == "yolo",\n        rect=args.family == "yolo",\n        multi_scale=0.0,\n        patience=args.localization_epochs,\n        lr0=args.localization_lr0,\n        optimizer="AdamW",\n        lrf=0.1,\n        cos_lr=True,\n        warmup_epochs=0.5,\n        close_mosaic=0,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        degrees=0.0,\n        translate=0.02,\n        scale=0.10,\n        fliplr=0.5,\n        flipud=0.0,\n        mosaic=0.0,\n        mixup=0.0,\n        copy_paste=0.0,\n        augmentations=[],\n        plots=False,\n        save=True,\n        verbose=True,\n        pretrained=True,\n    )\n    best = Path(result.save_dir) / "weights" / "best.pt"\n    if not best.exists():\n        raise FileNotFoundError(best)\n    return best\n\n\ndef predict(args, best, test_out, inference_geometry, tta):\n    model = model_class(args)(str(best))\n    results = model.predict(\n        source=str(test_out),\n        imgsz=args.imgsz,\n        conf=args.predict_conf,\n        iou=args.iou,\n        max_det=args.predict_max_det,\n        device=args.device,\n        augment=tta,\n        stream=True,\n        verbose=False,\n    )\n    rows = []\n    seen = set()\n    empty = []\n    for r in results:\n        seen.add(Path(r.path).stem)\n        if r.boxes is None or len(r.boxes) == 0:\n            empty.append(Path(r.path).stem)\n        stem = Path(r.path).stem\n        if r.boxes is None:\n            continue\n        xyxy = restore_boxes(\n            r.boxes.xyxy.detach().cpu().numpy(),\n            inference_geometry[stem],\n        )\n        confs = r.boxes.conf.detach().cpu().numpy()\n        classes = r.boxes.cls.detach().cpu().numpy().astype(int)\n        for box, score, cls in zip(xyxy, confs, classes):\n            x1, y1, x2, y2 = map(float, box)\n            if x2 <= x1 or y2 <= y1:\n                continue\n            rows.append(\n                {\n                    "image_id": image_id_value(stem),\n                    "class_id": int(cls),\n                    "confidence": float(score),\n                    "x1": x1,\n                    "y1": y1,\n                    "x2": x2,\n                    "y2": y2,\n                }\n            )\n\n    expected_ids = set(inference_geometry)\n    if seen != expected_ids:\n        raise RuntimeError("Some test images were not processed")\n    counts = pd.Series([row["image_id"] for row in rows]).value_counts()\n    (Path(args.workdir) / "prediction_audit.json").write_text(json.dumps({\n        "processed_images": len(seen),\n        "no_detection_ids": empty,\n        "prediction_confidence": args.predict_conf,\n        "prediction_max_det": args.predict_max_det,\n        "tta": tta,\n        "predictions_total": len(rows),\n        "predictions_per_image_min": int(counts.min()) if len(counts) else 0,\n        "predictions_per_image_median": float(counts.median()) if len(counts) else 0.0,\n        "predictions_per_image_max": int(counts.max()) if len(counts) else 0,\n    }, indent=2))\n    sub = pd.DataFrame(rows)\n    if sub.empty:\n        raise RuntimeError("No detections produced")\n    sub.insert(0, "id", np.arange(len(sub), dtype=int))\n    expected = ["id", "image_id", "class_id", "confidence", "x1", "y1", "x2", "y2"]\n    sub = sub[expected]\n    if args.sample_submission and pd.read_csv(args.sample_submission, nrows=0).columns.tolist() != expected:\n        raise ValueError("Sample submission columns do not match")\n    if not np.isfinite(sub[["confidence", "x1", "y1", "x2", "y2"]].to_numpy()).all():\n        raise ValueError("Non-finite submission values")\n    if not sub["id"].is_unique:\n        raise RuntimeError("Submission id is not unique")\n    if not sub["class_id"].between(0, 17).all():\n        raise RuntimeError("class_id outside 0..17")\n    if not sub["confidence"].between(0, 1).all():\n        raise RuntimeError("confidence outside 0..1")\n    if not ((sub["x2"] > sub["x1"]) & (sub["y2"] > sub["y1"])).all():\n        raise RuntimeError("Invalid boxes")\n    out_path = Path(args.workdir) / "submission.csv"\n    sub.to_csv(out_path, index=False)\n    return out_path\n\n\ndef unique_stems(files):\n    result = {}\n    for path in files:\n        if path.stem in result:\n            raise ValueError(f"Duplicate image or XML ID: {path.stem}")\n        result[path.stem] = path\n    return result\n\n\ndef model_class(args):\n    from ultralytics import RTDETR, YOLO\n\n    return RTDETR if args.family == "rtdetr" else YOLO\n\n\ndef metric_summary(metrics):\n    class_ids = [int(c) for c in metrics.box.ap_class_index]\n    per_class = {}\n    for position, class_id in enumerate(class_ids):\n        per_class[str(class_id)] = {\n            "ap50_95": float(metrics.box.ap[position]),\n            "ap50": float(metrics.box.ap50[position]),\n            "ap75": float(metrics.box.all_ap[position, 5]),\n            "precision": float(metrics.box.p[position]),\n            "recall": float(metrics.box.r[position]),\n        }\n    summary = {\n        "map50_95": float(metrics.box.map),\n        "map50": float(metrics.box.map50),\n        "map75": float(metrics.box.map75),\n        "localization_gap_map50_minus_map75": float(metrics.box.map50 - metrics.box.map75),\n        "per_class": per_class,\n        "worst_present_class_ap": min((row["ap50_95"] for row in per_class.values()), default=None),\n        "absent_class_ids": sorted(set(range(18)) - set(class_ids)),\n    }\n    matrix = np.asarray(metrics.confusion_matrix.matrix)\n    if matrix.shape == (19, 19) and matrix.sum() > 0:\n        true_positive = np.diag(matrix)[:18]\n        summary["false_positives_per_class"] = (\n            matrix[:18, :].sum(axis=1) - true_positive\n        ).astype(int).tolist()\n        summary["false_negatives_per_class"] = (\n            matrix[:, :18].sum(axis=0) - true_positive\n        ).astype(int).tolist()\n        summary["confusion_matrix_predicted_rows_true_columns"] = matrix.astype(int).tolist()\n    else:\n        summary["false_positives_per_class"] = None\n        summary["false_negatives_per_class"] = None\n        summary["confusion_matrix_predicted_rows_true_columns"] = None\n        summary["confusion_status"] = "unavailable; compute from saved predictions at an explicit operating point"\n    values = [summary["map50_95"], summary["map50"], summary["map75"]]\n    if not np.isfinite(values).all():\n        raise RuntimeError("Non-finite evaluation metrics")\n    return summary\n\n\ndef evaluate(args, best, yaml_path, splits, output_name, tta=False):\n    model = model_class(args)(str(best))\n    report = {\n        "checkpoint": str(best),\n        "sha256": hashlib.sha256(Path(best).read_bytes()).hexdigest(),\n        "tta": tta,\n    }\n    for split in splits:\n        metrics = model.val(data=str(yaml_path), split=split, imgsz=args.imgsz,\n                            batch=args.batch if args.batch > 0 else 4,\n                            device=args.device, workers=args.workers, conf=args.conf,\n                            max_det=args.max_det, iou=args.iou, plots=False,\n                            augment=tta,\n                            project=str(Path(args.workdir) / "evaluation"),\n                            name=f"{Path(output_name).stem}_{split}")\n        report[split] = metric_summary(metrics)\n        (Path(args.workdir) / output_name).write_text(json.dumps(report, indent=2))\n    return report\n\n\ndef promote_candidate(base_report, candidate_report, min_gain):\n    base = base_report["val"]\n    candidate = candidate_report["val"]\n    gain = candidate["map50_95"] - base["map50_95"]\n    map75_gain = candidate["map75"] - base["map75"]\n    worst_regression = candidate["worst_present_class_ap"] - base["worst_present_class_ap"]\n    passed = (gain >= min_gain or (gain >= 0 and map75_gain >= min_gain)) and worst_regression >= -0.03\n    return {\n        "passed": passed,\n        "map50_95_gain": gain,\n        "map75_gain": map75_gain,\n        "worst_present_class_ap_change": worst_regression,\n        "minimum_gain": min_gain,\n    }\n\n\ndef check_submission_gate(args, report):\n    holdout_map = report["test"]["map50_95"]\n    gate = {\n        "metric": "holdout_mAP50_95",\n        "value": holdout_map,\n        "minimum": args.min_holdout_map,\n        "passed": holdout_map >= args.min_holdout_map,\n    }\n    (Path(args.workdir) / "submission_gate.json").write_text(json.dumps(gate, indent=2))\n    return gate\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--train-cubes", required=True)\n    p.add_argument("--test-cubes", required=True)\n    p.add_argument("--ranking-cubes")\n    p.add_argument("--train-xml", required=True)\n    p.add_argument("--class-file", required=True)\n    p.add_argument("--workdir", default="/kaggle/working/hsi_submission01")\n    p.add_argument("--cache-dir")\n    p.add_argument("--groups-file")\n    p.add_argument("--split-manifest")\n    p.add_argument("--cv-folds", type=int, default=3)\n    p.add_argument("--cv-fold-index", type=int)\n    p.add_argument("--model", default="yolo26m.pt")\n    p.add_argument("--checkpoint")\n    p.add_argument("--epochs", type=int, default=120)\n    p.add_argument("--imgsz", type=int, default=1024)\n    p.add_argument("--workers", type=int, default=2)\n    p.add_argument("--device", default="0")\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--input-policy", choices=["auto", "native", "square_pad"], default="auto")\n    p.add_argument("--stem-init", choices=["random", "rgb_mean"], default="rgb_mean")\n    p.add_argument("--normalization", choices=["N0", "N1", "N2", "N3", "N4"], default="N0")\n    p.add_argument("--spectral-preprocess", choices=["SP0", "SP1", "SP2", "SP3"], default="SP0")\n    p.add_argument("--normalization-samples-per-image", type=int, default=256)\n    p.add_argument("--mnf-components", type=int)\n    p.add_argument("--savgol-window", type=int, default=5)\n    p.add_argument("--savgol-order", type=int, default=2)\n    p.add_argument("--multi-scale", type=float, default=0.0)\n    p.add_argument("--spectral-aug-copies", type=int, default=0)\n    p.add_argument("--spectral-aug-classes", type=parse_class_ids, default=[])\n    p.add_argument("--spectral-gain", type=float, default=0.05)\n    p.add_argument("--spectral-tilt", type=float, default=0.03)\n    p.add_argument("--spectral-noise", type=float, default=0.01)\n    p.add_argument("--localization-epochs", type=int, default=0)\n    p.add_argument("--localization-lr0", type=float, default=0.0001)\n    p.add_argument("--promotion-min-gain", type=float, default=0.005)\n    p.add_argument("--tta", action="store_true")\n    p.add_argument("--tta-min-gain", type=float, default=0.002)\n    p.add_argument("--design-only", action="store_true")\n    p.add_argument("--conf", type=float, default=0.001)\n    p.add_argument("--predict-conf", type=float, default=0.001)\n    p.add_argument("--iou", type=float, default=0.65)\n    p.add_argument("--max-det", type=int, default=300)\n    p.add_argument("--predict-max-det", type=int, default=300)\n    p.add_argument("--min-holdout-map", type=float, default=0.50)\n    p.add_argument("--family", choices=["yolo", "rtdetr"], default="yolo")\n    p.add_argument("--batch", type=int, default=-1)\n    p.add_argument("--run-name", default="submission01")\n    p.add_argument("--sample-submission")\n    p.add_argument("--evaluate-holdout", action="store_true")\n    args = p.parse_args()\n    if args.spectral_aug_copies < 0:\n        raise ValueError("spectral_aug_copies must be non-negative")\n    if args.spectral_preprocess == "SP1" and args.mnf_components is None:\n        raise ValueError("SP1 requires --mnf-components selected on validation")\n    if not 0 <= args.multi_scale <= 0.9:\n        raise ValueError("multi_scale must be in 0..0.9")\n    if args.tta and args.family != "yolo":\n        raise ValueError("Built-in validated TTA is enabled only for YOLO in this runner")\n    import torch\n    if args.device != "cpu" and not torch.cuda.is_available():\n        raise RuntimeError("GPU is unavailable. Enable a Kaggle GPU and keep its CUDA PyTorch installation.")\n\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    Path(args.workdir).mkdir(parents=True, exist_ok=True)\n\n    versions = {name: importlib.metadata.version(name) for name in ("torch", "ultralytics", "numpy")}\n    run_config = {\n        "args": vars(args),\n        "resolved_input_policy": resolve_input_policy(args.family, args.input_policy),\n        "versions": versions,\n    }\n    (Path(args.workdir) / "run_config.json").write_text(json.dumps(run_config, indent=2))\n    yaml_path, test_out, inference_geometry = build_dataset(args)\n    if args.checkpoint:\n        base_checkpoint = Path(args.checkpoint)\n        if not base_checkpoint.exists():\n            raise FileNotFoundError(base_checkpoint)\n    else:\n        base_checkpoint = train(args, yaml_path)\n    base_report = evaluate(\n        args,\n        base_checkpoint,\n        yaml_path,\n        ["val"],\n        "evaluation_base.json",\n    )\n    selected_checkpoint = base_checkpoint\n    selected_report = base_report\n    selection = {"base_checkpoint": str(base_checkpoint), "localization_finetune": None, "tta": None}\n    finetuned_checkpoint = localization_finetune(args, yaml_path, base_checkpoint)\n    if finetuned_checkpoint:\n        finetuned_report = evaluate(\n            args,\n            finetuned_checkpoint,\n            yaml_path,\n            ["val"],\n            "evaluation_localized.json",\n        )\n        decision = promote_candidate(base_report, finetuned_report, args.promotion_min_gain)\n        decision["checkpoint"] = str(finetuned_checkpoint)\n        selection["localization_finetune"] = decision\n        if decision["passed"]:\n            selected_checkpoint = finetuned_checkpoint\n            selected_report = finetuned_report\n    selected_tta = False\n    if args.tta:\n        tta_report = evaluate(\n            args,\n            selected_checkpoint,\n            yaml_path,\n            ["val"],\n            "evaluation_tta.json",\n            tta=True,\n        )\n        decision = promote_candidate(selected_report, tta_report, args.tta_min_gain)\n        selection["tta"] = decision\n        if decision["passed"]:\n            selected_report = tta_report\n            selected_tta = True\n    selection["selected_checkpoint"] = str(selected_checkpoint)\n    selection["selected_checkpoint_sha256"] = hashlib.sha256(Path(selected_checkpoint).read_bytes()).hexdigest()\n    selection["selected_tta"] = selected_tta\n    (Path(args.workdir) / "selection.json").write_text(json.dumps(selection, indent=2))\n    if args.design_only:\n        print(selection)\n        print("Design-only run complete. Holdout and competition test prediction were not used.")\n        return\n    holdout_report = evaluate(\n        args,\n        selected_checkpoint,\n        yaml_path,\n        ["test"],\n        "evaluation_holdout.json",\n        tta=selected_tta,\n    )\n    report = {\n        "checkpoint": str(selected_checkpoint),\n        "sha256": selection["selected_checkpoint_sha256"],\n        "tta": selected_tta,\n        "val": selected_report["val"],\n        "test": holdout_report["test"],\n    }\n    (Path(args.workdir) / "evaluation.json").write_text(json.dumps(report, indent=2))\n    gate = check_submission_gate(args, report)\n    print(gate)\n    if not gate["passed"]:\n        print("Submission blocked: holdout mAP50-95 is below minimum.")\n        print(selected_checkpoint)\n        return\n    submission = predict(args, selected_checkpoint, test_out, inference_geometry, selected_tta)\n    print(selected_checkpoint)\n    print(submission)\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/build_folds.py').write_text('import argparse\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport xml.etree.ElementTree as ET\n\nimport numpy as np\n\n\nEXCLUDED_IDS = ("1227", "1836", "1855")\nAREA_EDGES = (0, 32**2, 96**2)\n\n\ndef digest_file(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n\ndef unique_files(root, suffixes):\n    result = {}\n    for path in sorted(Path(root).rglob("*")):\n        if path.is_file() and path.suffix.lower() in suffixes:\n            if path.stem in result:\n                raise ValueError(f"Duplicate image ID {path.stem}: {result[path.stem]}, {path}")\n            result[path.stem] = path\n    if not result:\n        raise ValueError(f"No matching files in {root}")\n    return result\n\n\ndef non_object_metadata(root):\n    result = defaultdict(list)\n\n    def visit(node, prefix):\n        if node.tag.lower() == "object":\n            return\n        key = f"{prefix}/{node.tag}" if prefix else node.tag\n        result[key].append({"text": (node.text or "").strip(), "attributes": dict(node.attrib)})\n        for child in node:\n            visit(child, key)\n\n    visit(root, "")\n    return dict(result)\n\n\ndef read_annotation(path, names):\n    root = ET.parse(path).getroot()\n    width = float(root.findtext("size/width"))\n    height = float(root.findtext("size/height"))\n    if not np.isfinite([width, height]).all() or min(width, height) <= 0:\n        raise ValueError(f"Invalid original dimensions in {path}")\n    counts = np.zeros(len(names), dtype=np.int64)\n    areas = []\n    class_sizes = np.zeros((len(names), 3), dtype=np.int64)\n    clipped = discarded = 0\n    for obj in root.findall("object"):\n        name = (obj.findtext("name") or "").strip()\n        if name not in names:\n            raise ValueError(f"Unknown class {name!r} in {path}")\n        box = np.array([float(obj.findtext(f"bndbox/{key}")) for key in ("xmin", "ymin", "xmax", "ymax")])\n        if not np.isfinite(box).all():\n            raise ValueError(f"Non-finite box in {path}")\n        bounded = np.clip(box, 0, [width, height, width, height])\n        clipped += int(not np.array_equal(box, bounded))\n        x1, y1, x2, y2 = bounded\n        if x2 <= x1 or y2 <= y1:\n            discarded += 1\n            continue\n        area = float((x2 - x1) * (y2 - y1))\n        class_id = names.index(name)\n        counts[class_id] += 1\n        areas.append(area)\n        class_sizes[class_id, int(np.searchsorted(AREA_EDGES[1:], area, side="right"))] += 1\n    return {\n        "width": width, "height": height, "class_counts": counts.tolist(),\n        "class_presence": (counts > 0).astype(int).tolist(), "box_areas": areas,\n        "class_size_counts": class_sizes.tolist(), "size_counts": class_sizes.sum(0).tolist(),\n        "clipped_box_count": clipped, "discarded_box_count": discarded,\n        "metadata": non_object_metadata(root),\n    }\n\n\ndef load_groups(ids, groups_file=None, groups_evidence=None):\n    if groups_file is None:\n        if groups_evidence is not None:\n            raise ValueError("--groups-evidence requires --groups-file")\n        return {stem: stem for stem in ids}, {\n            "mode": "image_id_only",\n            "relationship_origin": "identity_fallback",\n            "source_grouping_verified": False,\n            "verification_status": "no_source_relationship_claimed",\n        }\n    if groups_evidence is None or not Path(groups_evidence).read_text().strip():\n        raise ValueError("A group mapping requires a non-empty --groups-evidence file documenting the relationship")\n    with Path(groups_file).open(newline="") as handle:\n        reader = csv.DictReader(handle)\n        if set(reader.fieldnames or []) != {"image_id", "group_id"}:\n            raise ValueError("Groups CSV must contain exactly image_id,group_id")\n        groups = {}\n        for row in reader:\n            stem, group = row["image_id"].strip(), row["group_id"].strip()\n            if not stem or not group or stem in groups:\n                raise ValueError("Empty or duplicate image ID/group in groups CSV")\n            groups[stem] = group\n    if set(ids) - groups.keys():\n        raise ValueError(f"Groups CSV missing IDs: {sorted(set(ids) - groups.keys())[:10]}")\n    return {stem: groups[stem] for stem in ids}, {\n        "mode": "provided_group_mapping",\n        "relationship_origin": "caller_provided_csv",\n        "source_grouping_verified": False,\n        "verification_status": "requires_evidence_review",\n        "groups_file_sha256": digest_file(groups_file),\n        "groups_evidence_sha256": digest_file(groups_evidence),\n        "groups_evidence": Path(groups_evidence).read_text(),\n        "unused_group_ids": sorted(set(groups) - set(ids)),\n    }\n\n\ndef check_split(split, ids, groups):\n    flat = [stem for part in ("train", "val", "test") for stem in split[part]]\n    if len(flat) != len(set(flat)) or set(flat) != set(ids):\n        raise ValueError("Split must cover every usable image exactly once")\n    seen = {}\n    for part in ("train", "val", "test"):\n        if not split[part]:\n            raise ValueError(f"Empty {part} partition")\n        for stem in split[part]:\n            group = groups[stem]\n            if group in seen and seen[group] != part:\n                raise ValueError(f"Group {group} leaks across partitions")\n            seen[group] = part\n\n\ndef assign_groups(ids, groups, records, seed):\n    members = defaultdict(list)\n    for stem in sorted(ids):\n        members[groups[stem]].append(stem)\n    if len(members) < 3:\n        raise ValueError("Three-fold CV requires at least three development groups")\n    keys = sorted(members)\n    vectors = np.asarray([\n        np.sum([records[s]["class_presence"] + records[s]["class_counts"] + records[s]["size_counts"] + [1] for s in members[g]], axis=0).tolist() + [1]\n        for g in keys\n    ], dtype=float)\n    total = vectors.sum(0)\n    active = total > 0\n    normalized = vectors[:, active] / total[active]\n    rng = np.random.default_rng(seed)\n    ties = rng.random(len(keys))\n    order = sorted(range(len(keys)), key=lambda i: (-float(np.max(normalized[i])), -float(np.sum(normalized[i])), ties[i]))\n    loads = np.zeros((3, normalized.shape[1]))\n    counts = np.zeros(3, dtype=int)\n    chosen = {}\n    priority = rng.permutation(3).tolist()\n    for i in order:\n        scores = []\n        for fold in range(3):\n            trial = loads.copy()\n            trial[fold] += normalized[i]\n            scores.append(float(np.square(trial - trial.mean(0)).sum()))\n        fold = min(range(3), key=lambda f: (scores[f], counts[f], priority.index(f)))\n        loads[fold] += normalized[i]\n        counts[fold] += len(members[keys[i]])\n        chosen[keys[i]] = fold\n    return {stem: chosen[groups[stem]] for stem in ids}\n\n\ndef summarize(ids, records, groups, names):\n    counts = np.sum([records[s]["class_counts"] for s in ids], axis=0).astype(int)\n    presence = np.sum([records[s]["class_presence"] for s in ids], axis=0).astype(int)\n    areas = [area for s in ids for area in records[s]["box_areas"]]\n    histogram, _ = np.histogram(areas, bins=[0, 256, 1024, 4096, 9216, 16384, 65536, np.inf])\n    return {\n        "image_count": len(ids), "group_count": len({groups[s] for s in ids}),\n        "object_count": int(counts.sum()), "class_object_counts": dict(zip(names, counts.tolist())),\n        "class_image_counts": dict(zip(names, presence.tolist())),\n        "missing_classes": [name for name, count in zip(names, counts) if count == 0],\n        "size_counts": dict(zip(("small", "medium", "large"), np.sum([records[s]["size_counts"] for s in ids], axis=0).astype(int).tolist())),\n        "class_size_counts": dict(zip(names, np.sum([records[s]["class_size_counts"] for s in ids], axis=0).astype(int).tolist())),\n        "box_area_histogram": {"edges": [0, 256, 1024, 4096, 9216, 16384, 65536, "infinity"], "counts": histogram.tolist()},\n    }\n\n\ndef build_folds(xml_dir, cube_dir, class_file, legacy_split, output_dir, groups_file=None, groups_evidence=None, seed=42):\n    names = [line.strip() for line in Path(class_file).read_text().splitlines() if line.strip()]\n    if not names or len(names) != len(set(names)):\n        raise ValueError("Class names must be nonempty and unique")\n    xmls = unique_files(xml_dir, {".xml"})\n    cubes = unique_files(cube_dir, {".png", ".npy", ".npz", ".tif", ".tiff"})\n    if set(xmls) != set(cubes):\n        raise ValueError("Annotation and cube image IDs differ")\n    ids = sorted(set(xmls) - set(EXCLUDED_IDS))\n    legacy = json.loads(Path(legacy_split).read_text())\n    if set(legacy) != {"train", "val", "test"}:\n        raise ValueError("Legacy split requires train, val, test")\n    legacy = {key: list(map(str, value)) for key, value in legacy.items()}\n    check_split(legacy, ids, {stem: stem for stem in ids})\n    records = {stem: read_annotation(xmls[stem], names) for stem in ids}\n    groups, group_info = load_groups(ids, groups_file, groups_evidence)\n    holdout_groups = {groups[stem] for stem in legacy["test"]}\n    holdout = sorted(stem for stem in ids if groups[stem] in holdout_groups)\n    development = sorted(set(ids) - set(holdout))\n    assignments = assign_groups(development, groups, records, seed)\n    folds = []\n    for index in range(3):\n        split = {"train": [s for s in development if assignments[s] != index], "val": [s for s in development if assignments[s] == index], "test": holdout}\n        check_split(split, ids, groups)\n        folds.append(split)\n    if sorted(s for split in folds for s in split["val"]) != development:\n        raise AssertionError("Development images must validate exactly once")\n    fields = defaultdict(Counter)\n    inventory = {}\n    for stem, record in records.items():\n        for key, values in record["metadata"].items():\n            fields[key].update(json.dumps(v, sort_keys=True) for v in values)\n        inventory[stem] = {\n            "xml": str(xmls[stem].relative_to(xml_dir)), "cube": str(cubes[stem].relative_to(cube_dir)),\n            "xml_sha256": digest_file(xmls[stem]), "cube_size_bytes": cubes[stem].stat().st_size,\n            "xml_metadata": record["metadata"], "original_width": record["width"], "original_height": record["height"],\n            "original_box_areas": record["box_areas"], "clipped_box_count": record["clipped_box_count"],\n            "discarded_box_count": record["discarded_box_count"],\n        }\n    grouping_fields = [key for key in fields if any(word in key.lower() for word in ("capture", "session", "source", "folder", "sequence", "scene", "video"))]\n    candidate_fields = [key for key in grouping_fields if len(fields[key]) > 1]\n    non_informative_fields = [key for key in grouping_fields if len(fields[key]) <= 1]\n    config = {"seed": seed, "cv_folds": 3, "excluded_ids": list(EXCLUDED_IDS), "area_bins": list(AREA_EDGES),\n              "legacy_split_sha256": digest_file(legacy_split), "class_file_sha256": digest_file(class_file),\n              "group_mapping_sha256": digest_json(groups), "annotation_inventory_sha256": digest_json(inventory),\n              "groups_evidence_sha256": group_info.get("groups_evidence_sha256"),\n              "algorithm": "seeded_greedy_multilabel_presence_object_counts_original_area_bins_image_and_group_counts"}\n    audit = {\n        "status": "folds_built_not_training_results", "code_hash": digest_file(__file__), "config": config,\n        "config_hash": digest_json(config), "groups": group_info,\n        "excluded_ids": {stem: {"present": stem in xmls,\n                                "reason": "Existing runner exclusion retained for comparability; the historical mismatch rationale was not re-verified",\n                                "evidence_status": "historical_policy_unverified"} for stem in EXCLUDED_IDS},\n        "holdout": {"status": "previously_inspected", "legacy_count": len(legacy["test"]), "group_closed_count": len(holdout),\n                    "added_for_group_closure": sorted(set(holdout) - set(legacy["test"])),\n                    "previously_trained_by_legacy_checkpoint": sorted(set(holdout) & set(legacy["train"])),\n                    "legacy_checkpoint_declared_group_overlap": sorted(holdout_groups & {groups[s] for s in legacy["train"]}),\n                    "use": "Freeze new designs before evaluation; cannot claim an untouched final holdout. Existing checkpoints cannot provide new CV evidence."},\n        "metadata_audit": {"non_object_fields": {key: {"unique_values": len(values), "value_counts": dict(sorted(values.items()))} for key, values in sorted(fields.items())},\n                           "candidate_group_fields": candidate_fields,\n                           "non_informative_group_fields": non_informative_fields,\n                           "unresolved_metadata_candidates": candidate_fields if groups_file is None else [],\n                           "xml_folders": dict(Counter(str(path.parent.relative_to(xml_dir)) for path in xmls.values())),\n                           "cube_folders": dict(Counter(str(path.parent.relative_to(cube_dir)) for path in cubes.values())),\n                           "numeric_filename_count": sum(stem.isdigit() for stem in ids),\n                           "numeric_name_policy": "Numbers alone are not capture/session evidence; no numeric or visual grouping inferred",\n                           "conclusion": ("No varying official capture/session/source field exists; image_id_only is the supported fallback" if not candidate_fields else "Only image identities are currently supported; varying source/session candidates require provenance review") if groups_file is None else "Caller-provided mapping used for isolation; no source relationship is claimed until attached provenance is reviewed"},\n        "coordinate_policy": "Original XML pixel coordinates; clip to XML image bounds; discard nonpositive boxes as existing runner does; no resize before areas",\n        "assertions": {"image_disjoint": True, "declared_group_mapping_disjoint": True,\n                       "verified_source_group_disjoint": True if group_info["source_grouping_verified"] else None,\n                       "all_usable_ids_covered": True, "validation_once_per_development_id": True,\n                       "legacy_holdout_preserved": True},\n        "folds": [{"fold": i, "split_hash": digest_json(split), "partitions": {part: summarize(stems, records, groups, names) for part, stems in split.items()}} for i, split in enumerate(folds)],\n    }\n    out = Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    paths = [out / f"fold{i}.json" for i in range(3)] + [out / "groups.csv", out / "fold_audit.json", out / "metadata_inventory.json"]\n    if any(path.exists() for path in paths):\n        raise FileExistsError("Use a fresh output directory; frozen fold artifacts cannot be overwritten")\n    for i, split in enumerate(folds):\n        (out / f"fold{i}.json").write_text(json.dumps(split, indent=2) + "\\n")\n    with (out / "groups.csv").open("w", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["image_id", "group_id"])\n        writer.writerows((stem, groups[stem]) for stem in ids)\n    (out / "fold_audit.json").write_text(json.dumps(audit, indent=2, allow_nan=False) + "\\n")\n    (out / "metadata_inventory.json").write_text(json.dumps(inventory, indent=2, allow_nan=False) + "\\n")\n    return audit\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--xml-dir", required=True)\n    parser.add_argument("--cube-dir", required=True)\n    parser.add_argument("--class-file", required=True)\n    parser.add_argument("--legacy-split", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--groups-file")\n    parser.add_argument("--groups-evidence")\n    parser.add_argument("--seed", type=int, default=42)\n    audit = build_folds(**vars(parser.parse_args()))\n    print(json.dumps({"status": audit["status"], "grouping": audit["groups"]["mode"], "fold_sizes": [{part: value["image_count"] for part, value in fold["partitions"].items()} for fold in audit["folds"]], "holdout": audit["holdout"]["status"]}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/normalization.py').write_text('import json\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\n\n\nN_BANDS = 16\nMETHODS = ("N0", "N1", "N2", "N3", "N4")\n\n\n@dataclass(frozen=True)\nclass NormalizationState:\n    method: str\n    parameters: dict\n    fit_scope: str = "training_fold"\n    n_bands: int = N_BANDS\n    numeric_storage: str = "float32"\n    version: int = 1\n\n    def to_dict(self):\n        return asdict(self)\n\n    @classmethod\n    def from_dict(cls, value):\n        state = cls(**value)\n        _validate_state(state)\n        return state\n\n\ndef _validate_array(values, name="spectra"):\n    array = np.asarray(values)\n    if array.ndim < 2 or array.shape[-1] != N_BANDS:\n        raise ValueError(f"{name} must have 16 bands on its last axis")\n    if not np.issubdtype(array.dtype, np.number):\n        raise TypeError(f"{name} must be numeric")\n    if not np.isfinite(array).all():\n        raise ValueError(f"{name} must contain only finite values")\n    return array\n\n\ndef _training_matrix(training_data):\n    if isinstance(training_data, np.ndarray):\n        arrays = [training_data]\n    else:\n        arrays = list(training_data)\n    if not arrays:\n        raise ValueError("training_data must not be empty")\n    flattened = [_validate_array(array, "training_data").reshape(-1, N_BANDS) for array in arrays]\n    matrix = np.concatenate(flattened, axis=0).astype(np.float64, copy=False)\n    if matrix.shape[0] < 2:\n        raise ValueError("training_data must contain at least two spectral vectors")\n    return matrix\n\n\ndef _per_band_percentiles(matrix, percentiles):\n    low, high = np.percentile(matrix, percentiles, axis=0)\n    if np.any(high <= low):\n        raise ValueError("Every band must have a positive fitted range")\n    return low, high\n\n\ndef fit_normalization(\n    training_data: np.ndarray | Iterable[np.ndarray],\n    method: str,\n    clip_percentiles=(0.5, 99.5),\n    robust_percentiles=(2.0, 98.0),\n    epsilon=1e-6,\n):\n    method = method.upper()\n    if method not in METHODS:\n        raise ValueError(f"method must be one of {METHODS}")\n    if not np.isfinite(epsilon) or epsilon <= 0:\n        raise ValueError("epsilon must be finite and positive")\n    matrix = _training_matrix(training_data)\n    parameters = {"epsilon": float(epsilon)}\n    if method == "N1":\n        mean = matrix.mean(axis=0)\n        scale = matrix.std(axis=0)\n        scale = np.maximum(scale, epsilon)\n        parameters.update(mean=mean.tolist(), scale=scale.tolist())\n    else:\n        percentiles = robust_percentiles if method == "N2" else clip_percentiles\n        low, high = _per_band_percentiles(matrix, percentiles)\n        parameters.update(\n            clip_low=low.tolist(),\n            clip_high=high.tolist(),\n            percentiles=[float(percentiles[0]), float(percentiles[1])],\n        )\n        if method == "N2":\n            parameters["robust_definition"] = (\n                f"training-fold per-band {float(percentiles[0]):g}th/{float(percentiles[1]):g}th percentile"\n            )\n        if method == "N4":\n            parameters["area_definition"] = "sum across the 16 equally spaced bands"\n    state = NormalizationState(method=method, parameters=parameters)\n    _validate_state(state)\n    return state\n\n\ndef apply_normalization(values, state: NormalizationState):\n    _validate_state(state)\n    source = _validate_array(values).astype(np.float64, copy=True)\n    parameters = state.parameters\n    epsilon = float(parameters["epsilon"])\n    if state.method == "N1":\n        result = (source - np.asarray(parameters["mean"])) / np.asarray(parameters["scale"])\n    else:\n        low = np.asarray(parameters["clip_low"])\n        high = np.asarray(parameters["clip_high"])\n        clipped = np.clip(source, low, high)\n        if state.method in ("N0", "N2"):\n            result = (clipped - low) / (high - low)\n        elif state.method == "N3":\n            vector_mean = clipped.mean(axis=-1, keepdims=True)\n            vector_scale = clipped.std(axis=-1, keepdims=True)\n            result = (clipped - vector_mean) / np.maximum(vector_scale, epsilon)\n        else:\n            area = clipped.sum(axis=-1, keepdims=True)\n            safe = np.abs(area) > epsilon\n            result = np.divide(clipped, area, out=np.zeros_like(clipped), where=safe)\n    if not np.isfinite(result).all():\n        raise FloatingPointError("Normalization produced non-finite values")\n    return result.astype(np.float32)\n\n\ndef save_normalization_state(state: NormalizationState, path):\n    _validate_state(state)\n    Path(path).write_text(json.dumps(state.to_dict(), indent=2, allow_nan=False) + "\\n")\n\n\ndef load_normalization_state(path):\n    return NormalizationState.from_dict(json.loads(Path(path).read_text()))\n\n\ndef _validate_state(state):\n    if state.method not in METHODS:\n        raise ValueError(f"Unknown normalization method: {state.method}")\n    if state.fit_scope != "training_fold" or state.n_bands != N_BANDS:\n        raise ValueError("Normalization state must be fitted on a 16-band training fold")\n    parameters = state.parameters\n    if not np.isfinite(float(parameters.get("epsilon", np.nan))) or float(parameters["epsilon"]) <= 0:\n        raise ValueError("Normalization epsilon must be finite and positive")\n    required = ("mean", "scale") if state.method == "N1" else ("clip_low", "clip_high")\n    for key in required:\n        values = np.asarray(parameters.get(key), dtype=np.float64)\n        if values.shape != (N_BANDS,) or not np.isfinite(values).all():\n            raise ValueError(f"Normalization parameter {key} must contain 16 finite values")\n    if state.method == "N1" and np.any(np.asarray(parameters["scale"], dtype=np.float64) <= 0):\n        raise ValueError("Normalization scales must be positive")\n    if state.method != "N1" and np.any(\n        np.asarray(parameters["clip_high"], dtype=np.float64)\n        <= np.asarray(parameters["clip_low"], dtype=np.float64)\n    ):\n        raise ValueError("Normalization clip ranges must be positive")\n')
Path('/kaggle/working/mnf.py').write_text('import json\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport numpy as np\n\n\nN_BANDS = 16\n\n\n@dataclass(frozen=True)\nclass MNFState:\n    mean: list\n    components: list\n    inverse_components: list\n    eigenvalues: list\n    signal_covariance: list\n    noise_covariance: list\n    noise_estimation: str\n    fit_scope: str = "training_fold"\n    n_bands: int = N_BANDS\n    version: int = 1\n\n    def to_dict(self):\n        return asdict(self)\n\n    @classmethod\n    def from_dict(cls, value):\n        state = cls(**value)\n        _state_arrays(state)\n        return state\n\n\ndef _as_cubes(training_data):\n    if isinstance(training_data, np.ndarray):\n        if training_data.ndim in (2, 3):\n            cubes = [training_data]\n        elif training_data.ndim == 4:\n            cubes = list(training_data)\n        else:\n            raise ValueError("training_data must contain spectral matrices or cubes")\n    else:\n        cubes = list(training_data)\n    if not cubes:\n        raise ValueError("training_data must not be empty")\n    validated = []\n    for cube in cubes:\n        array = np.asarray(cube, dtype=np.float64)\n        if array.ndim not in (2, 3) or array.shape[-1] != N_BANDS:\n            raise ValueError("Every training item must have 16 bands on its last axis")\n        if not np.isfinite(array).all():\n            raise ValueError("training_data must contain only finite values")\n        validated.append(array)\n    return validated\n\n\ndef _noise_differences(cubes):\n    differences = []\n    for cube in cubes:\n        axes = (0,) if cube.ndim == 2 else (0, 1)\n        for axis in axes:\n            if cube.shape[axis] > 1:\n                differences.append(np.diff(cube, axis=axis).reshape(-1, N_BANDS))\n    if not differences:\n        raise ValueError("MNF noise estimation requires adjacent training samples")\n    return np.concatenate(differences, axis=0)\n\n\ndef fit_mnf(training_data, regularization=1e-6):\n    if not np.isfinite(regularization) or regularization <= 0:\n        raise ValueError("regularization must be finite and positive")\n    cubes = _as_cubes(training_data)\n    matrix = np.concatenate([cube.reshape(-1, N_BANDS) for cube in cubes], axis=0)\n    if matrix.shape[0] <= N_BANDS:\n        raise ValueError("MNF fitting requires more training vectors than spectral bands")\n    differences = _noise_differences(cubes)\n    mean = matrix.mean(axis=0)\n    signal_covariance = np.cov(matrix, rowvar=False)\n    noise_covariance = np.cov(differences, rowvar=False) / 2.0\n    if not np.isfinite(signal_covariance).all() or not np.isfinite(noise_covariance).all():\n        raise ValueError("MNF covariance estimation requires enough varying training samples")\n    floor = max(float(np.trace(noise_covariance)) / N_BANDS * regularization, regularization)\n    noise_covariance = noise_covariance + np.eye(N_BANDS) * floor\n    noise_values, noise_vectors = np.linalg.eigh(noise_covariance)\n    noise_values = np.maximum(noise_values, floor)\n    whitening = noise_vectors @ np.diag(1.0 / np.sqrt(noise_values))\n    whitened_signal = whitening.T @ signal_covariance @ whitening\n    eigenvalues, rotations = np.linalg.eigh(whitened_signal)\n    order = np.argsort(eigenvalues)[::-1]\n    components = whitening @ rotations[:, order]\n    inverse_components = np.linalg.inv(components)\n    state = MNFState(\n        mean=mean.tolist(),\n        components=components.tolist(),\n        inverse_components=inverse_components.tolist(),\n        eigenvalues=eigenvalues[order].tolist(),\n        signal_covariance=signal_covariance.tolist(),\n        noise_covariance=noise_covariance.tolist(),\n        noise_estimation="half covariance of horizontal and vertical adjacent training-fold differences",\n    )\n    _state_arrays(state)\n    return state\n\n\ndef transform_mnf(values, state: MNFState):\n    mean, components, _, _ = _state_arrays(state)\n    array = _validate_values(values)\n    transformed = (array.astype(np.float64, copy=False) - mean) @ components\n    return transformed.astype(np.float32)\n\n\ndef reconstruct_mnf(values, state: MNFState, n_components):\n    if not isinstance(n_components, int) or isinstance(n_components, bool) or not 1 <= n_components <= N_BANDS:\n        raise ValueError("n_components must be an integer from 1 to 16 selected on validation")\n    mean, components, inverse_components, _ = _state_arrays(state)\n    array = _validate_values(values)\n    scores = (array.astype(np.float64, copy=False) - mean) @ components\n    scores[..., n_components:] = 0.0\n    reconstructed = scores @ inverse_components + mean\n    if not np.isfinite(reconstructed).all():\n        raise FloatingPointError("MNF reconstruction produced non-finite values")\n    return reconstructed.astype(np.float32)\n\n\ndef reconstruction_error(original, reconstructed):\n    original_array = _validate_values(original).astype(np.float64, copy=False)\n    reconstructed_array = _validate_values(reconstructed).astype(np.float64, copy=False)\n    if original_array.shape != reconstructed_array.shape:\n        raise ValueError("original and reconstructed spectra must have the same shape")\n    difference = reconstructed_array - original_array\n    return {\n        "rmse": float(np.sqrt(np.mean(difference ** 2))),\n        "mae": float(np.mean(np.abs(difference))),\n    }\n\n\ndef save_mnf_state(state: MNFState, path):\n    _state_arrays(state)\n    Path(path).write_text(json.dumps(state.to_dict(), indent=2, allow_nan=False) + "\\n")\n\n\ndef load_mnf_state(path):\n    return MNFState.from_dict(json.loads(Path(path).read_text()))\n\n\ndef _validate_values(values):\n    array = np.asarray(values)\n    if array.ndim < 2 or array.shape[-1] != N_BANDS:\n        raise ValueError("Spectra must have 16 bands on their last axis")\n    if not np.isfinite(array).all():\n        raise ValueError("Spectra must contain only finite values")\n    return array\n\n\ndef _state_arrays(state):\n    if state.fit_scope != "training_fold" or state.n_bands != N_BANDS:\n        raise ValueError("MNF state must be fitted on a 16-band training fold")\n    mean = np.asarray(state.mean, dtype=np.float64)\n    components = np.asarray(state.components, dtype=np.float64)\n    inverse = np.asarray(state.inverse_components, dtype=np.float64)\n    eigenvalues = np.asarray(state.eigenvalues, dtype=np.float64)\n    signal_covariance = np.asarray(state.signal_covariance, dtype=np.float64)\n    noise_covariance = np.asarray(state.noise_covariance, dtype=np.float64)\n    if mean.shape != (N_BANDS,) or components.shape != (N_BANDS, N_BANDS):\n        raise ValueError("Invalid MNF state dimensions")\n    if inverse.shape != components.shape or eigenvalues.shape != (N_BANDS,):\n        raise ValueError("Invalid MNF inverse or eigenvalue dimensions")\n    if signal_covariance.shape != components.shape or noise_covariance.shape != components.shape:\n        raise ValueError("Invalid MNF covariance dimensions")\n    if not all(\n        np.isfinite(value).all()\n        for value in (mean, components, inverse, eigenvalues, signal_covariance, noise_covariance)\n    ):\n        raise ValueError("MNF state must contain only finite values")\n    return mean, components, inverse, eigenvalues\n')
Path('/kaggle/working/spectral_filters.py').write_text('import json\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.signal import savgol_filter\n\n\nN_BANDS = 16\n\n\n@dataclass(frozen=True)\nclass MSCState:\n    reference_spectrum: list\n    fit_scope: str = "training_fold"\n    n_bands: int = N_BANDS\n    reference_method: str = "mean of training-fold spectral vectors"\n    version: int = 1\n\n    def to_dict(self):\n        return asdict(self)\n\n    @classmethod\n    def from_dict(cls, value):\n        state = cls(**value)\n        _reference(state)\n        return state\n\n\ndef savitzky_golay(values, window_length=5, polyorder=2):\n    array = _validate_values(values)\n    if not isinstance(window_length, int) or isinstance(window_length, bool):\n        raise TypeError("window_length must be an integer")\n    if window_length < 3 or window_length > 15 or window_length % 2 == 0:\n        raise ValueError("window_length must be an odd integer from 3 to 15")\n    if not isinstance(polyorder, int) or isinstance(polyorder, bool) or not 0 <= polyorder < window_length:\n        raise ValueError("polyorder must be a non-negative integer smaller than window_length")\n    result = savgol_filter(\n        array.astype(np.float64, copy=True),\n        window_length=window_length,\n        polyorder=polyorder,\n        axis=-1,\n        mode="interp",\n    )\n    return result.astype(np.float32)\n\n\ndef fit_msc(training_data):\n    if isinstance(training_data, np.ndarray):\n        arrays = [training_data]\n    else:\n        arrays = list(training_data)\n    if not arrays:\n        raise ValueError("training_data must not be empty")\n    matrix = np.concatenate([_validate_values(array).reshape(-1, N_BANDS) for array in arrays], axis=0)\n    reference = matrix.astype(np.float64, copy=False).mean(axis=0)\n    if np.var(reference) <= 0:\n        raise ValueError("MSC reference spectrum must vary across bands")\n    return MSCState(reference_spectrum=reference.tolist())\n\n\ndef apply_msc(values, state: MSCState, epsilon=1e-8):\n    if not np.isfinite(epsilon) or epsilon <= 0:\n        raise ValueError("epsilon must be finite and positive")\n    array = _validate_values(values).astype(np.float64, copy=True)\n    reference = _reference(state)\n    centered_reference = reference - reference.mean()\n    denominator = float(centered_reference @ centered_reference)\n    flat = array.reshape(-1, N_BANDS)\n    slopes = ((flat - flat.mean(axis=1, keepdims=True)) @ centered_reference) / denominator\n    intercepts = flat.mean(axis=1) - slopes * reference.mean()\n    stable = np.abs(slopes) > epsilon\n    corrected = flat.copy()\n    corrected[stable] = (flat[stable] - intercepts[stable, None]) / slopes[stable, None]\n    corrected = corrected.reshape(array.shape)\n    if not np.isfinite(corrected).all():\n        raise FloatingPointError("MSC produced non-finite values")\n    return corrected.astype(np.float32)\n\n\ndef save_msc_state(state: MSCState, path):\n    _reference(state)\n    Path(path).write_text(json.dumps(state.to_dict(), indent=2, allow_nan=False) + "\\n")\n\n\ndef load_msc_state(path):\n    return MSCState.from_dict(json.loads(Path(path).read_text()))\n\n\ndef _validate_values(values):\n    array = np.asarray(values)\n    if array.ndim < 2 or array.shape[-1] != N_BANDS:\n        raise ValueError("Spectra must have 16 bands on their last axis")\n    if not np.issubdtype(array.dtype, np.number) or not np.isfinite(array).all():\n        raise ValueError("Spectra must contain only finite numeric values")\n    return array\n\n\ndef _reference(state):\n    if state.fit_scope != "training_fold" or state.n_bands != N_BANDS:\n        raise ValueError("MSC state must be fitted on a 16-band training fold")\n    reference = np.asarray(state.reference_spectrum, dtype=np.float64)\n    if reference.shape != (N_BANDS,) or not np.isfinite(reference).all() or np.var(reference) <= 0:\n        raise ValueError("MSC reference must contain 16 finite, non-constant values")\n    return reference\n')
Path('/kaggle/working/pipeline.py').write_text('from dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport json\n\nimport numpy as np\n\nif __package__:\n    from .mnf import MNFState, fit_mnf, reconstruct_mnf\n    from .normalization import NormalizationState, apply_normalization, fit_normalization\n    from .spectral_filters import MSCState, apply_msc, fit_msc, savitzky_golay\nelse:\n    from mnf import MNFState, fit_mnf, reconstruct_mnf\n    from normalization import NormalizationState, apply_normalization, fit_normalization\n    from spectral_filters import MSCState, apply_msc, fit_msc, savitzky_golay\n\n\n@dataclass(frozen=True)\nclass SpectralPipelineState:\n    normalization: dict\n    spectral_method: str\n    spectral_parameters: dict\n    signed_encoding: dict\n    fit_scope: str = "training_fold"\n    n_bands: int = 16\n    version: int = 1\n\n    def to_dict(self):\n        return asdict(self)\n\n    @classmethod\n    def from_dict(cls, value):\n        state = cls(**value)\n        _validate_state(state)\n        return state\n\n\ndef fit_pipeline(\n    training_cubes,\n    normalization_method,\n    spectral_method="SP0",\n    seed=42,\n    samples_per_cube=256,\n    mnf_components=None,\n    savgol_window=5,\n    savgol_order=2,\n):\n    spectral_method = spectral_method.upper()\n    if spectral_method not in ("SP0", "SP1", "SP2", "SP3"):\n        raise ValueError("spectral_method must be SP0, SP1, SP2, or SP3")\n    if samples_per_cube < 32:\n        raise ValueError("samples_per_cube must be at least 32")\n    rng = np.random.default_rng(seed)\n    sampled = []\n    patches = []\n    for source in training_cubes:\n        cube = np.asarray(source, dtype=np.float32)\n        if cube.ndim != 3 or cube.shape[-1] != 16 or not np.isfinite(cube).all():\n            raise ValueError("Each training cube must be a finite H x W x 16 array")\n        flat = cube.reshape(-1, 16)\n        count = min(samples_per_cube, len(flat))\n        sampled.append(flat[rng.choice(len(flat), count, replace=False)])\n        if spectral_method == "SP1":\n            side = max(2, int(np.sqrt(count)))\n            side = min(side, cube.shape[0], cube.shape[1])\n            top = int(rng.integers(0, cube.shape[0] - side + 1))\n            left = int(rng.integers(0, cube.shape[1] - side + 1))\n            patches.append(cube[top : top + side, left : left + side])\n    if not sampled:\n        raise ValueError("training_cubes must not be empty")\n    sample = np.concatenate(sampled)\n    parameters = {\n        "seed": int(seed),\n        "samples_per_cube": int(samples_per_cube),\n        "training_cube_count": len(sampled),\n    }\n    if spectral_method == "SP1":\n        if mnf_components is None:\n            raise ValueError("SP1 requires validation-selected mnf_components")\n        mnf = fit_mnf(patches)\n        parameters.update(mnf_state=mnf.to_dict(), mnf_components=int(mnf_components))\n        transformed = reconstruct_mnf(sample, mnf, int(mnf_components))\n    elif spectral_method == "SP2":\n        parameters.update(window_length=int(savgol_window), polyorder=int(savgol_order))\n        transformed = savitzky_golay(sample, window_length=savgol_window, polyorder=savgol_order)\n    elif spectral_method == "SP3":\n        msc = fit_msc(sample)\n        parameters.update(msc_state=msc.to_dict())\n        transformed = apply_msc(sample, msc)\n    else:\n        transformed = sample\n    normalization = fit_normalization(transformed, normalization_method)\n    normalized = apply_normalization(transformed, normalization)\n    encoding = _fit_encoding(normalized, normalization.method)\n    state = SpectralPipelineState(\n        normalization=normalization.to_dict(),\n        spectral_method=spectral_method,\n        spectral_parameters=parameters,\n        signed_encoding=encoding,\n    )\n    _validate_state(state)\n    return state\n\n\ndef apply_pipeline(cube, state, output="float32"):\n    _validate_state(state)\n    array = np.asarray(cube, dtype=np.float32)\n    if array.ndim < 2 or array.shape[-1] != 16 or not np.isfinite(array).all():\n        raise ValueError("cube must be a finite array with 16 bands on its last axis")\n    parameters = state.spectral_parameters\n    if state.spectral_method == "SP1":\n        transformed = reconstruct_mnf(array, MNFState.from_dict(parameters["mnf_state"]), int(parameters["mnf_components"]))\n    elif state.spectral_method == "SP2":\n        transformed = savitzky_golay(array, window_length=int(parameters["window_length"]), polyorder=int(parameters["polyorder"]))\n    elif state.spectral_method == "SP3":\n        transformed = apply_msc(array, MSCState.from_dict(parameters["msc_state"]))\n    else:\n        transformed = array\n    normalized = apply_normalization(transformed, NormalizationState.from_dict(state.normalization))\n    unit = _encode(normalized, state.signed_encoding)\n    if output == "float32":\n        return unit.astype(np.float32)\n    if output == "uint8":\n        return np.rint(unit * 255).astype(np.uint8)\n    raise ValueError("output must be float32 or uint8")\n\n\ndef save_pipeline_state(state, path):\n    _validate_state(state)\n    Path(path).write_text(json.dumps(state.to_dict(), indent=2, allow_nan=False) + "\\n")\n\n\ndef load_pipeline_state(path):\n    return SpectralPipelineState.from_dict(json.loads(Path(path).read_text()))\n\n\ndef _fit_encoding(normalized, method):\n    if method in ("N0", "N2"):\n        return {"method": "unit_interval", "zero_encoded_as": 0.0, "fit_scope": "training_fold"}\n    if method in ("N1", "N3"):\n        bound = max(float(np.percentile(np.abs(normalized), 99.5)), 1e-6)\n        return {"method": "symmetric_signed", "bound": bound, "zero_encoded_as": 0.5, "fit_scope": "training_fold"}\n    upper = max(float(np.percentile(normalized, 99.5)), 1e-6)\n    lower = min(0.0, float(np.percentile(normalized, 0.5)))\n    return {"method": "global_affine", "lower": lower, "upper": upper, "zero_encoded_as": float(-lower / (upper - lower)), "fit_scope": "training_fold"}\n\n\ndef _encode(values, encoding):\n    if encoding["method"] == "unit_interval":\n        result = np.clip(values, 0, 1)\n    elif encoding["method"] == "symmetric_signed":\n        bound = float(encoding["bound"])\n        result = np.clip(values, -bound, bound) / (2 * bound) + 0.5\n    else:\n        lower, upper = float(encoding["lower"]), float(encoding["upper"])\n        result = np.clip((values - lower) / (upper - lower), 0, 1)\n    if not np.isfinite(result).all():\n        raise FloatingPointError("Storage encoding produced non-finite values")\n    return result\n\n\ndef _validate_state(state):\n    if state.fit_scope != "training_fold" or state.n_bands != 16:\n        raise ValueError("Pipeline state must be fitted on a 16-band training fold")\n    NormalizationState.from_dict(state.normalization)\n    if state.spectral_method not in ("SP0", "SP1", "SP2", "SP3"):\n        raise ValueError("Unknown spectral method")\n    method = state.signed_encoding.get("method")\n    if method not in ("unit_interval", "symmetric_signed", "global_affine"):\n        raise ValueError("Unknown storage encoding")\n    if state.signed_encoding.get("fit_scope") != "training_fold":\n        raise ValueError("Storage encoding must be training-fold fitted")\n    if method == "symmetric_signed" and float(state.signed_encoding.get("bound", 0)) <= 0:\n        raise ValueError("Signed encoding bound must be positive")\n    if method == "global_affine" and float(state.signed_encoding.get("upper", 0)) <= float(state.signed_encoding.get("lower", 0)):\n        raise ValueError("Global encoding range must be positive")\n')


In [ ]:
import hashlib
import json
from pathlib import Path
import subprocess
import sys

data_matches = list(Path('/kaggle/input').rglob('hyperspectral-2026'))
legacy_matches = list(Path('/kaggle/input').rglob('hsi_plan_c_ratio/split_manifest.json'))
assert len(data_matches) == 1, data_matches
assert len(legacy_matches) == 1, legacy_matches
base = data_matches[0]
fold_dir = Path('/kaggle/working/folds')
subprocess.run([
    sys.executable, '/kaggle/working/build_folds.py',
    '--xml-dir', str(base / 'data_train/data_train/Annotations/VIS'),
    '--cube-dir', str(base / 'data_train/data_train/VIS'),
    '--class-file', str(base / 'class.txt'),
    '--legacy-split', str(legacy_matches[0]),
    '--output-dir', str(fold_dir),
    '--seed', '42',
], check=True)
work = Path('/kaggle/working/p1-img1024-f0')
command = [
    sys.executable, '/kaggle/working/hsi_runner.py',
    '--train-cubes', str(base / 'data_train/data_train/VIS'),
    '--test-cubes', str(base / 'data_test/data_test/VIS'),
    '--train-xml', str(base / 'data_train/data_train/Annotations/VIS'),
    '--class-file', str(base / 'class.txt'),
    '--workdir', str(work),
    '--cache-dir', '/kaggle/temp/p1-img1024-f0',
    '--groups-file', str(fold_dir / 'groups.csv'),
    '--split-manifest', str(fold_dir / 'fold0.json'),
    '--family', 'rtdetr', '--model', 'rtdetr-l.pt',
    '--epochs', '10', '--imgsz', '1024', '--batch', '2', '--workers', '2',
    '--device', '0', '--seed', '42', '--input-policy', 'square_pad', '--stem-init', 'random',
    '--run-name', 'p1-img1024-f0', '--conf', '0.001', '--iou', '0.65', '--max-det', '300',
    '--design-only',
] + []
subprocess.run(command, check=True)
evaluation = json.loads((work / 'evaluation_base.json').read_text())
record = {
    'experiment_id': 'p1-img1024-f0',
    'screen_budget_epochs': 10,
    'fold': 0,
    'split_hash': hashlib.sha256((fold_dir / 'fold0.json').read_bytes()).hexdigest(),
    'code_hash': hashlib.sha256(Path('/kaggle/working/hsi_runner.py').read_bytes()).hexdigest(),
    'evaluation': evaluation,
    'submission_created': False,
}
(work / 'screen_evidence.json').write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))
